In [1]:
print("HELLO")

HELLO


In [3]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

text = """Sarah is an employee at prismaticAI, a leading technology company based in Westside Valley. She has been working there for the past three years as a software engineer.
Michael is also an employee at prismaticAI, where he works as a data scientist. He joined the company two years ago after completing his graduate studies.
prismaticAI is a well-known technology company that specializes in developing cutting-edge software solutions and artificial intelligence applications. The company has a diverse workforce of talented individuals from various backgrounds.
Both Sarah and Michael are highly skilled professionals who contribute significantly to prismaticAI's success. They work closely with their respective teams to develop innovative products and services that meet the evolving needs of the company's clients."""

# loader = TextLoader(text)
# documents = loader.load()
text_splitter = RecursiveCharacterTextSplitter()
texts = text_splitter.split_text(text=text)

docs = [Document(page_content=text) for text in texts]

In [4]:
texts

["Sarah is an employee at prismaticAI, a leading technology company based in Westside Valley. She has been working there for the past three years as a software engineer.\nMichael is also an employee at prismaticAI, where he works as a data scientist. He joined the company two years ago after completing his graduate studies.\nprismaticAI is a well-known technology company that specializes in developing cutting-edge software solutions and artificial intelligence applications. The company has a diverse workforce of talented individuals from various backgrounds.\nBoth Sarah and Michael are highly skilled professionals who contribute significantly to prismaticAI's success. They work closely with their respective teams to develop innovative products and services that meet the evolving needs of the company's clients."]

In [14]:
docs

[Document(metadata={}, page_content="Sarah is an employee at prismaticAI, a leading technology company based in Westside Valley. She has been working there for the past three years as a software engineer.\nMichael is also an employee at prismaticAI, where he works as a data scientist. He joined the company two years ago after completing his graduate studies.\nprismaticAI is a well-known technology company that specializes in developing cutting-edge software solutions and artificial intelligence applications. The company has a diverse workforce of talented individuals from various backgrounds.\nBoth Sarah and Michael are highly skilled professionals who contribute significantly to prismaticAI's success. They work closely with their respective teams to develop innovative products and services that meet the evolving needs of the company's clients.")]

In [18]:
from langchain_groq import ChatGroq
from langchain_experimental.graph_transformers import LLMGraphTransformer
import getpass
import os
from dotenv import load_dotenv
load_dotenv()

key = os.getenv("GROQ_API_KEY")


llm = ChatGroq(model="openai/gpt-oss-120b", )


llm_transformer = LLMGraphTransformer(llm=llm)
graph_documents = llm_transformer.convert_to_graph_documents(docs)

In [19]:
graph_documents

[GraphDocument(nodes=[Node(id='Sarah', type='Person', properties={}), Node(id='Michael', type='Person', properties={}), Node(id='Prismaticai', type='Company', properties={}), Node(id='Westside Valley', type='Location', properties={}), Node(id='Software Engineer', type='Role', properties={}), Node(id='Data Scientist', type='Role', properties={})], relationships=[Relationship(source=Node(id='Sarah', type='Person', properties={}), target=Node(id='Prismaticai', type='Company', properties={}), type='EMPLOYEE', properties={}), Relationship(source=Node(id='Michael', type='Person', properties={}), target=Node(id='Prismaticai', type='Company', properties={}), type='EMPLOYEE', properties={}), Relationship(source=Node(id='Prismaticai', type='Company', properties={}), target=Node(id='Westside Valley', type='Location', properties={}), type='LOCATED_IN', properties={}), Relationship(source=Node(id='Sarah', type='Person', properties={}), target=Node(id='Software Engineer', type='Role', properties={})

In [ ]:
import 

In [24]:
from langchain_neo4j import Neo4jGraph


graph_store = Neo4jGraph(url="neo4j://127.0.0.1:7687", username="neo4j", password="Tjtk2004!", database="test")
graph_store.add_graph_documents(graph_documents)

In [27]:
# pip install llama-index-graph-stores-neo4j

from llama_index.graph_stores.neo4j import Neo4jGraphStore
from llama_index.core import StorageContext
from llama_index.core.query_engine import RetrieverQueryEngine
from llama_index.core.retrievers import KnowledgeGraphRAGRetriever



storage_context = StorageContext.from_defaults(graph_store=graph_store)

graph_rag_retriever = KnowledgeGraphRAGRetriever(
    storage_context=storage_context,
    verbose=True,
)

query_engine = RetrieverQueryEngine.from_args(graph_rag_retriever)

C:\Users\tejas\AppData\Local\Temp\ipykernel_9660\2628622222.py:12: DeprecationWarning: Call to deprecated class KnowledgeGraphRAGRetriever. (KnowledgeGraphRAGRetriever is deprecated, it is recommended to use PropertyGraphIndex and associated retrievers instead.) -- Deprecated since version 0.10.53.
  graph_rag_retriever = KnowledgeGraphRAGRetriever(


In [7]:
"""
Pydantic models for each node type in the resume knowledge graph.

These mirror the ALLOWED_NODES schema used in the LLMGraphTransformer
pipeline (resume_graph_pipeline_hardened.py). Use these when you want
structured, validated extraction instead of -- or alongside -- the
LLMGraphTransformer's free-form graph output. Handy for:
  - Validating LLM JSON output before writing to Neo4j
  - Type-safe access when reading nodes back out of the graph
  - A stricter alternative extraction path (structured output via
    llm.with_structured_output(...)) if LLMGraphTransformer keeps
    producing inconsistent shapes.
"""

from __future__ import annotations
from typing import Optional
from pydantic import BaseModel, Field


class Person(BaseModel):
    id: str = Field(..., description="Full name, e.g. 'Tejas Kadam'")
    email: Optional[str] = None
    phone: Optional[str] = None
    github: Optional[str] = None
    linkedin: Optional[str] = None
    leetcode: Optional[str] = None
    summary: Optional[str] = Field(None, description="Professional summary blurb")


class Institution(BaseModel):
    id: str = Field(..., description="Institution name, e.g. 'Vellore Institute of Technology'")
    location: Optional[str] = None


class Degree(BaseModel):
    id: str = Field(..., description="Degree name, e.g. 'B.Tech Computer Science and Business Systems'")
    institution_id: str = Field(..., description="id of the Institution this degree was earned at")
    start_year: Optional[int] = None
    end_year: Optional[int] = None
    cgpa: Optional[float] = None
    percentage: Optional[float] = None


class Skill(BaseModel):
    id: str = Field(..., description="Skill/tool/technology name, e.g. 'LangChain'")
    category: Optional[str] = Field(
        None, description="e.g. 'Language', 'Framework', 'Concept', 'Tool'"
    )


class Project(BaseModel):
    id: str = Field(..., description="Project name, e.g. 'NeuralNote'")
    description: Optional[str] = Field(None, description="1-2 sentence summary")
    tech_stack: list[str] = Field(default_factory=list, description="Skill ids used in this project")
    github_url: Optional[str] = None


class Certification(BaseModel):
    id: str = Field(..., description="Certification name")
    organization_id: str = Field(..., description="id of the Organization that issued it")
    certificate_code: Optional[str] = None
    hours: Optional[int] = None


class Organization(BaseModel):
    id: str = Field(..., description="Organization name, e.g. 'IBM'")


class Achievement(BaseModel):
    id: str = Field(..., description="Achievement description, e.g. 'Hackathon Finalist (3x)'")
    metric: Optional[str] = Field(None, description="Numeric or percentile value if applicable")


class ResumeGraph(BaseModel):
    """Top-level container -- the full structured extraction for one resume."""
    person: Person
    institutions: list[Institution] = Field(default_factory=list)
    degrees: list[Degree] = Field(default_factory=list)
    skills: list[Skill] = Field(default_factory=list)
    projects: list[Project] = Field(default_factory=list)
    certifications: list[Certification] = Field(default_factory=list)
    organizations: list[Organization] = Field(default_factory=list)
    achievements: list[Achievement] = Field(default_factory=list)

In [ ]:
"""
Resume -> Neo4j pipeline using structured output (Pydantic schema) instead of
LLMGraphTransformer's free-form JSON extraction.

Why this instead of LLMGraphTransformer:
  - The model's output is validated against the ResumeGraph schema before
    you ever touch it -- malformed/truncated output raises a clear
    validation error instead of the opaque "Failed to parse tool call
    arguments as JSON" BadRequestError you hit earlier.
  - Relationships (Degree -> Institution, Certification -> Organization)
    are enforced by the schema itself (institution_id / organization_id
    fields), not left to the LLM to remember to emit as separate edges.
  - Writing to Neo4j is done with explicit, predictable MERGE statements
    per node type, so it's easy to see exactly what graph shape you get --
    no surprises from the LLM inventing extra node/relationship types.

Install:
  pip install langchain-groq neo4j python-dotenv pydantic
"""

import os
from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_neo4j import Neo4jGraph



load_dotenv()


# ---------------------------------------------------------------------------
# 1. Structured extraction
# ---------------------------------------------------------------------------
def extract_resume(resume_text: str, llm) -> ResumeGraph:
    structured_llm = llm.with_structured_output(ResumeGraph)
    prompt = (
        "Extract all information from this resume into the given schema. "
        "Use the person's exact name (as it appears at the top of the resume) "
        "as the Person id. Every Degree must reference a valid institution_id "
        "that also appears in the institutions list. Every Certification must "
        "reference a valid organization_id that also appears in the "
        "organizations list. Every Project's tech_stack must only contain "
        "skill ids that also appear in the skills list. Do not invent "
        "information that isn't in the resume text.\n\n"
        f"Resume:\n{resume_text}"
    )
    result = structured_llm.invoke(prompt)
    if not isinstance(result, ResumeGraph):
        # some provider/model combos return a dict instead of the model instance
        result = ResumeGraph.model_validate(result)
    return result


# ---------------------------------------------------------------------------
# 2. Write to Neo4j -- explicit MERGE per node type, then relationships.
#    MERGE (not CREATE) makes every write idempotent: re-running this on
#    the same resume updates existing nodes instead of duplicating them.
# ---------------------------------------------------------------------------
def write_resume_graph(graph: Neo4jGraph, data: ResumeGraph):
    # Person
    graph.query(
        """
        MERGE (p:Person {id: $id})
        SET p.email = $email, p.phone = $phone, p.github = $github,
            p.linkedin = $linkedin, p.leetcode = $leetcode, p.summary = $summary
        """,
        params=data.person.model_dump(),
    )

    # Institutions
    for inst in data.institutions:
        graph.query(
            """
            MERGE (i:Institution {id: $id})
            SET i.location = $location
            WITH i
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:STUDIED_AT]->(i)
            """,
            params={**inst.model_dump(), "person_id": data.person.id},
        )

    # Degrees
    for deg in data.degrees:
        graph.query(
            """
            MERGE (d:Degree {id: $id})
            SET d.start_year = $start_year, d.end_year = $end_year,
                d.cgpa = $cgpa, d.percentage = $percentage
            WITH d
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:HOLDS_DEGREE]->(d)
            WITH d
            MATCH (i:Institution {id: $institution_id})
            MERGE (d)-[:AT_INSTITUTION]->(i)
            """,
            params={**deg.model_dump(), "person_id": data.person.id},
        )

    # Skills
    for skill in data.skills:
        graph.query(
            """
            MERGE (s:Skill {id: $id})
            SET s.category = $category
            WITH s
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:HAS_SKILL]->(s)
            """,
            params={**skill.model_dump(), "person_id": data.person.id},
        )

    # Projects (+ tech_stack edges to existing Skill nodes)
    for proj in data.projects:
        graph.query(
            """
            MERGE (pr:Project {id: $id})
            SET pr.description = $description, pr.github_url = $github_url
            WITH pr
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:BUILT]->(pr)
            WITH pr
            UNWIND $tech_stack AS tech_id
            MATCH (s:Skill {id: tech_id})
            MERGE (pr)-[:USES_TECHNOLOGY]->(s)
            """,
            params={**proj.model_dump(), "person_id": data.person.id},
        )

    # Organizations
    for org in data.organizations:
        graph.query("MERGE (o:Organization {id: $id})", params=org.model_dump())

    # Certifications
    for cert in data.certifications:
        graph.query(
            """
            MERGE (c:Certification {id: $id})
            SET c.certificate_code = $certificate_code, c.hours = $hours
            WITH c
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:EARNED_CERTIFICATION]->(c)
            WITH c
            MATCH (o:Organization {id: $organization_id})
            MERGE (c)-[:ISSUED_BY]->(o)
            """,
            params={**cert.model_dump(), "person_id": data.person.id},
        )

    # Achievements
    for ach in data.achievements:
        graph.query(
            """
            MERGE (a:Achievement {id: $id})
            SET a.metric = $metric
            WITH a
            MATCH (p:Person {id: $person_id})
            MERGE (p)-[:ACHIEVED]->(a)
            """,
            params={**ach.model_dump(), "person_id": data.person.id},
        )


# ---------------------------------------------------------------------------
# 3. Validation -- same connectivity check as before, still worth running
# ---------------------------------------------------------------------------
def validate_graph(graph: Neo4jGraph, person_id: str):
    dupes = graph.query("MATCH (p:Person) RETURN p.id AS id")
    if len(dupes) > 1:
        print(f"WARNING: found {len(dupes)} Person nodes, expected 1:")
        for r in dupes:
            print(f"  - {r['id']}")

    orphans = graph.query(
        """
        MATCH (p:Person {id: $id})
        CALL (p) {
          MATCH (p)-[*]-(reachable)
          RETURN collect(DISTINCT reachable) AS reached
        }
        MATCH (n)
        WHERE NOT n IN reached AND n <> p AND NOT n:Person
        RETURN labels(n) AS labels, n.id AS id
        """,
        params={"id": person_id},
    )
    if orphans:
        print(f"WARNING: {len(orphans)} node(s) not connected to Person:")
        for r in orphans:
            print(f"  - {r['labels']}: {r['id']}")
    else:
        print("Graph connectivity OK -- all nodes reachable from Person.")


# ---------------------------------------------------------------------------
# 4. Full run
# ---------------------------------------------------------------------------
if __name__ == "__main__":
    llm = ChatGroq(model="openai/gpt-oss-120b", max_tokens=8000)

    graph = Neo4jGraph(
        url=os.environ.get("NEO4J_URI", "neo4j://127.0.0.1:7687"),
        username=os.environ.get("NEO4J_USERNAME", "neo4j"),
        password="Tjtk2004!",
        database=os.environ.get("NEO4J_DATABASE", "test"),
        refresh_schema=False,
    )

    resume_text =   # your PDF-extracted text[0].page_content goes here

    data = extract_resume(resume_text, llm)
    print(data.model_dump_json(indent=2))  # inspect before writing, if you want

    write_resume_graph(graph, data)
    validate_graph(graph, data.person.id)

BadRequestError: Error code: 400 - {'error': {'message': 'Tool choice is required, but model did not call a tool', 'type': 'invalid_request_error', 'code': 'tool_use_failed', 'failed_generation': 'I’m ready to extract the information, but I need the full text of the resume to do so. Could you please provide the complete resume content?'}}